In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from rich import print

from app.graph.state import AssistantState
from langchain_core.messages import HumanMessage
from app.graph.node import route_node, health_node, should_continue
from app.tools.health_tools import log_workout

def build_assit_graph():
    graph = StateGraph(AssistantState)

    graph.add_node("route", route_node)
    graph.add_node("health", health_node)
    graph.add_node("tools", ToolNode([log_workout])) #-> LOGs IN DB

    graph.add_edge(START, "route")
    graph.add_edge("route", "health")
    graph.add_conditional_edges(
        "health",
        should_continue,
        {
            "tools": "tools",
            "end": END
        }
    )
    graph.add_edge("tools", "health")

    return graph.compile()


graph = build_assit_graph()
print(graph.get_graph().draw_mermaid())